In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Check GPU enabled
import torch
import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from pathlib import Path
import json
import random
print(torch.cuda.is_available())

True


In [5]:
# Load the sequences and labels for training
loaded_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {loaded_sequences_df.shape}")

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (630987, 2)


In [6]:
train_df, test_df = train_test_split(loaded_sequences_df, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")

Training set size: 504789
Testing set size: 126198


In [7]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1, dtype=np.float32)
    vec2 = np.asarray(vec2, dtype=np.float32)
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0
    return float(dot_product / (norm_vec1 * norm_vec2))

def features_to_vector(features):
    features = np.asarray(features, dtype=np.float32)
    if features.ndim == 1:
        features = features.reshape(1, -1)
    summary_vector = np.concatenate([
        features.mean(axis=0),
        features.std(axis=0),
        features.min(axis=0),
        features.max(axis=0)
    ])
    return summary_vector.astype(np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [8]:
# Prepare normalized tensors
train_sequences = np.stack(train_df['sequence'].to_numpy()).astype(np.float32)
test_sequences  = np.stack(test_df['sequence'].to_numpy()).astype(np.float32)

feature_mean = train_sequences.reshape(-1, train_sequences.shape[-1]).mean(axis=0)
feature_std  = train_sequences.reshape(-1, train_sequences.shape[-1]).std(axis=0) + 1e-6

X_train = (train_sequences - feature_mean) / feature_std
X_test  = (test_sequences  - feature_mean) / feature_std

label_encoder = LabelEncoder()
label_encoder.fit(loaded_sequences_df['label'])
y_train = label_encoder.transform(train_df['label'])
y_test  = label_encoder.transform(test_df['label'])

input_dim       = X_train.shape[-1]   # 3
sequence_length = X_train.shape[1]    # 256
num_classes     = len(label_encoder.classes_)

print(f'Train tensor shape : {X_train.shape}')
print(f'Test tensor shape  : {X_test.shape}')
print(f'Input dimension    : {input_dim}')
print(f'Sequence length    : {sequence_length}')
print(f'Classes            : {num_classes}')

Train tensor shape : (504789, 256, 3)
Test tensor shape  : (126198, 256, 3)
Input dimension    : 3
Sequence length    : 256
Classes            : 289


### Siamese CNN Model
Metric learning with triplet loss — learns an embedding space where sequences from the same MIDI file cluster together and sequences from different files are pushed apart.

**Architecture:**
```
Input (batch, 256, 3)
  -> Transpose (batch, 3, 256)
  -> Conv1D(3->64, k=5) + BatchNorm + ReLU + MaxPool2
  -> Conv1D(64->128, k=3) + BatchNorm + ReLU + MaxPool2
  -> Conv1D(128->256, k=3) + BatchNorm + ReLU
  -> Global Average Pool -> (batch, 256)
  -> FC(256->128) -> L2-normalize -> (batch, 128)
```
**Loss:** TripletLoss (margin=0.2) with semi-hard negative mining

**Retrieval:** cosine similarity against prototype embeddings (mean-pooled per class)

In [9]:
# ----- Siamese CNN -----
class SiameseCNN(nn.Module):
    """
    1D-CNN that maps a note sequence to an L2-normalised 128-dim embedding.
    Shared weights are used for anchor, positive, and negative in triplet training.
    """
    def __init__(self, input_dim=3, embedding_dim=128,
                 conv_channels=(64, 128, 256),
                 kernel_sizes=(5, 3, 3), dropout=0.3):
        super().__init__()

        # Conv Block 1: (batch, input_dim, 256) -> (batch, 64, 128)
        self.conv1 = nn.Conv1d(input_dim,          conv_channels[0], kernel_sizes[0], padding='same')
        self.bn1   = nn.BatchNorm1d(conv_channels[0])
        self.pool1 = nn.MaxPool1d(2)

        # Conv Block 2: (batch, 64, 128) -> (batch, 128, 64)
        self.conv2 = nn.Conv1d(conv_channels[0], conv_channels[1], kernel_sizes[1], padding='same')
        self.bn2   = nn.BatchNorm1d(conv_channels[1])
        self.pool2 = nn.MaxPool1d(2)

        # Conv Block 3: (batch, 128, 64) -> (batch, 256, 64)
        self.conv3 = nn.Conv1d(conv_channels[1], conv_channels[2], kernel_sizes[2], padding='same')
        self.bn3   = nn.BatchNorm1d(conv_channels[2])

        # Global Average Pool + FC
        self.gap     = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(conv_channels[2], embedding_dim)

    def forward(self, x, return_embedding=False):
        """
        x: (batch, seq_len, input_dim)
        returns: L2-normalised embedding (batch, 128)
        """
        x = x.transpose(1, 2)           # -> (batch, input_dim, seq_len)

        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = F.relu(self.bn3(self.conv3(x)))

        x = self.gap(x).squeeze(-1)      # -> (batch, 256)
        x = self.dropout(x)
        embedding = self.fc(x)           # -> (batch, 128)
        embedding = F.normalize(embedding, p=2, dim=1)

        if return_embedding:
            # Signature compatible with GRU notebook (logits, embedding)
            return embedding, embedding
        return embedding


# ----- Triplet Loss -----
class TripletLoss(nn.Module):
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        pos_dist = F.pairwise_distance(anchor, positive, p=2)
        neg_dist = F.pairwise_distance(anchor, negative, p=2)
        losses   = F.relu(pos_dist - neg_dist + self.margin)
        metrics  = {
            'pos_dist_mean'      : pos_dist.mean().item(),
            'neg_dist_mean'      : neg_dist.mean().item(),
            'num_active_triplets': (losses > 0).sum().item(),
            'margin_violations'  : (pos_dist > neg_dist).sum().item(),
        }
        return losses.mean(), metrics


# ----- Semi-Hard Negative Mining -----
def mine_semi_hard_triplets(embeddings, labels):
    """
    For each anchor, select a random positive (same label) and
    a semi-hard negative: d(a,n) > d(a,p) but within margin.
    Falls back to hardest negative if no semi-hard exists.
    Returns (anchors, positives, negatives) or (None, None, None).
    """
    batch_size  = embeddings.size(0)
    dist_matrix = torch.cdist(embeddings, embeddings, p=2)
    anchors, positives, negatives = [], [], []

    for i in range(batch_size):
        anchor_label = labels[i]
        pos_mask = (labels == anchor_label) & (
            torch.arange(batch_size, device=labels.device) != i)
        neg_mask = labels != anchor_label
        if not pos_mask.any() or not neg_mask.any():
            continue

        pos_indices = pos_mask.nonzero(as_tuple=True)[0]
        pos_idx     = pos_indices[torch.randint(len(pos_indices), (1,)).item()]
        pos_dist    = dist_matrix[i, pos_idx]

        neg_indices = neg_mask.nonzero(as_tuple=True)[0]
        neg_dists   = dist_matrix[i, neg_indices]
        semi_hard   = neg_dists > pos_dist

        if semi_hard.any():
            sh_indices = neg_indices[semi_hard]
            neg_idx    = sh_indices[torch.randint(len(sh_indices), (1,)).item()]
        else:
            neg_idx = neg_indices[neg_dists.argmin()]

        anchors.append(embeddings[i])
        positives.append(embeddings[pos_idx])
        negatives.append(embeddings[neg_idx])

    if not anchors:
        return None, None, None
    return torch.stack(anchors), torch.stack(positives), torch.stack(negatives)


# ----- Triplet Dataset -----
class TripletSequenceDataset(Dataset):
    """Wraps (sequences, labels) arrays into a PyTorch Dataset."""
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels    = torch.tensor(labels,    dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [10]:
# Hyperparameters
cnn_embedding_dim = 128
cnn_hidden_dim    = 128   # kept for metadata parity with GRU notebook
cnn_learning_rate = 1e-3
cnn_epochs        = 5
cnn_batch_size    = 64
cnn_margin        = 0.2
cnn_dropout       = 0.3

In [11]:
# DataLoaders
train_dataset = TripletSequenceDataset(X_train, y_train)
test_dataset  = TripletSequenceDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=cnn_batch_size, shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=cnn_batch_size, shuffle=False)

print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')

# Model, loss, optimizer, scheduler
cnn_model     = SiameseCNN(input_dim=input_dim,
                            embedding_dim=cnn_embedding_dim,
                            dropout=cnn_dropout).to(device)
cnn_criterion = TripletLoss(margin=cnn_margin)
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=cnn_learning_rate)
cnn_scheduler = optim.lr_scheduler.StepLR(cnn_optimizer, step_size=2, gamma=0.5)

n_params = sum(p.numel() for p in cnn_model.parameters())
print(f'SiameseCNN parameters: {n_params:,}')
print(cnn_model)

Train batches: 7887  Test batches: 1972
SiameseCNN parameters: 158,080
SiameseCNN(
  (conv1): Conv1d(3, 64, kernel_size=(5,), stride=(1,), padding=same)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=same)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=same)
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gap): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=128, bias=True)
)


In [13]:
# ----- Training Loop -----
cnn_history = []

for epoch in range(cnn_epochs):

    # ── Train ──────────────────────────────────────────────────────────────
    cnn_model.train()
    train_loss, train_active, train_batches = 0.0, 0, 0

    for batch_seqs, batch_labels in train_loader:
        batch_seqs   = batch_seqs.to(device)
        batch_labels = batch_labels.to(device)

        embeddings = cnn_model(batch_seqs)
        anchors, positives, negatives = mine_semi_hard_triplets(embeddings, batch_labels)
        if anchors is None:
            continue

        loss, metrics = cnn_criterion(anchors, positives, negatives)
        cnn_optimizer.zero_grad()
        loss.backward()
        cnn_optimizer.step()

        train_loss   += loss.item()
        train_active += metrics['num_active_triplets']
        train_batches += 1

    cnn_scheduler.step()
    avg_train_loss = train_loss   / max(train_batches, 1)
    avg_active     = train_active / max(train_batches, 1)

    # ── Evaluate: build prototype embeddings from test set ──────────────────
    cnn_model.eval()
    embedding_sums   = np.zeros((num_classes, cnn_embedding_dim), dtype=np.float32)
    embedding_counts = np.zeros(num_classes, dtype=np.int32)
    all_embs, all_lbls = [], []

    with torch.no_grad():
        for batch_seqs, batch_labels in test_loader:
            embs = cnn_model(batch_seqs.to(device)).cpu().numpy()
            lbls = batch_labels.numpy()
            all_embs.append(embs)
            all_lbls.append(lbls)
            for emb, lbl in zip(embs, lbls):
                embedding_sums[lbl]   += emb
                embedding_counts[lbl] += 1

    all_embs = np.vstack(all_embs)
    all_lbls = np.concatenate(all_lbls)
    prototype_embeddings = embedding_sums / np.maximum(embedding_counts[:, None], 1)

    # Nearest-prototype Top-1 accuracy
    preds = np.array([
        int(np.argmax([cosine_similarity(emb, prototype_embeddings[c])
                       for c in range(num_classes)]))
        for emb in all_embs
    ])
    test_accuracy = float((preds == all_lbls).mean())

    epoch_metrics = {
        'epoch'          : epoch + 1,
        'train_loss'     : avg_train_loss,
        'active_triplets': avg_active,
        'test_accuracy'  : test_accuracy,
        'lr'             : cnn_scheduler.get_last_lr()[0],
    }
    cnn_history.append(epoch_metrics)
    print(
        f"CNN Epoch {epoch+1}/{cnn_epochs} - "
        f"train_loss: {avg_train_loss:.4f}, "
        f"active_triplets: {avg_active:.1f}, "
        f"test_accuracy: {test_accuracy:.4f}"
    )

cnn_history_df = pd.DataFrame(cnn_history)
print('\nSiameseCNN training complete.')
cnn_history_df

CNN Epoch 1/5 - train_loss: 0.0223, active_triplets: 4.1, test_accuracy: 0.0879
CNN Epoch 2/5 - train_loss: 0.0160, active_triplets: 3.2, test_accuracy: 0.1451
CNN Epoch 3/5 - train_loss: 0.0121, active_triplets: 2.5, test_accuracy: 0.1832
CNN Epoch 4/5 - train_loss: 0.0102, active_triplets: 2.2, test_accuracy: 0.2125
CNN Epoch 5/5 - train_loss: 0.0087, active_triplets: 1.9, test_accuracy: 0.2516

SiameseCNN training complete.


,epoch,train_loss,active_triplets,test_accuracy,lr
0,1,0.022331,4.092938,0.087862,0.00100
1,2,0.016020,3.174084,0.145105,0.00050
2,3,0.012149,2.526563,0.183236,0.00050
3,4,0.010230,2.200837,0.212476,0.00025
4,5,0.008734,1.878534,0.251620,0.00025


In [14]:
# Top-K accuracy and MRR on the internal test set
K_VALUES     = [1, 3, 5]
correct_at_k = {k: 0 for k in K_VALUES}
recip_ranks  = []

cnn_model.eval()
with torch.no_grad():
    for batch_seqs, batch_labels in test_loader:
        embs = cnn_model(batch_seqs.to(device)).cpu().numpy()
        for emb, lbl in zip(embs, batch_labels.numpy()):
            sims  = np.array([cosine_similarity(emb, prototype_embeddings[c])
                              for c in range(num_classes)])
            order = np.argsort(-sims)
            rank  = int(np.where(order == lbl)[0][0]) + 1
            recip_ranks.append(1.0 / rank)
            for k in K_VALUES:
                if rank <= k:
                    correct_at_k[k] += 1

n_test = len(recip_ranks)
print(f'Test set retrieval metrics ({n_test} sequences):')
for k in K_VALUES:
    print(f'  Top-{k} accuracy : {correct_at_k[k]/n_test:.4f}')
print(f'  MRR              : {np.mean(recip_ranks):.4f}')

Test set retrieval metrics (126198 sequences):
  Top-1 accuracy : 0.2516
  Top-3 accuracy : 0.4808
  Top-5 accuracy : 0.6005
  MRR              : 0.4118


In [15]:
# Sample cosine similarity inspection — same interface as GRU notebook
sample_index      = 0
sample_sequence   = torch.tensor(X_test[sample_index:sample_index+1],
                                  dtype=torch.float32).to(device)
sample_true_label = label_encoder.inverse_transform([y_test[sample_index]])[0]

with torch.no_grad():
    sample_embedding = cnn_model(sample_sequence).squeeze(0).cpu().numpy()

similarity_rows = [
    {'label': class_name,
     'cosine_similarity': cosine_similarity(sample_embedding, prototype_embeddings[c])}
    for c, class_name in enumerate(label_encoder.classes_)
]

similarity_df = (pd.DataFrame(similarity_rows)
                   .sort_values('cosine_similarity', ascending=False)
                   .head(5)
                   .reset_index(drop=True))

pred_label = similarity_df.iloc[0]['label']

print(f'Sample true label     : {sample_true_label}')
print(f'Sample predicted label: {pred_label}')
similarity_df

Sample true label     : burg_spinnerlied.mid
Sample predicted label: ty_februar.mid


,label,cosine_similarity
0,ty_februar.mid,0.923404
1,chp_op18.mid,0.918580
2,chpn_op10_e12.mid,0.880308
3,beethoven_hammerklavier_1.mid,0.842683
4,mendel_op19_5.mid,0.835695


In [16]:
# Save model checkpoint
cnn_model_save_path = '/content/drive/MyDrive/capstone_team_3/models/siamese_cnn_classifier.pth'
torch.save(cnn_model.state_dict(), cnn_model_save_path)
print(f'SiameseCNN model saved to {cnn_model_save_path}')

SiameseCNN model saved to /content/drive/MyDrive/capstone_team_3/models/siamese_cnn_classifier.pth


In [17]:
# Load model state for inference
cnn_model.load_state_dict(torch.load(cnn_model_save_path))
cnn_model.to(device)
cnn_model.eval()

SiameseCNN(
  (conv1): Conv1d(3, 64, kernel_size=(5,), stride=(1,), padding=same)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=same)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=same)
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gap): AdaptiveAvgPool1d(output_size=1)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=128, bias=True)
)

In [41]:
# Build corpus prototype embeddings from the full training set
# These act as the retrieval database for external test inference
cnn_model.eval()
corpus_sums   = np.zeros((num_classes, cnn_embedding_dim), dtype=np.float32)
corpus_counts = np.zeros(num_classes, dtype=np.int32)

with torch.no_grad():
    for batch_seqs, batch_labels in train_loader:
        embs = cnn_model(batch_seqs.to(device)).cpu().numpy()
        for emb, lbl in zip(embs, batch_labels.numpy()):
            corpus_sums[lbl]   += emb
            corpus_counts[lbl] += 1

corpus_prototype_embeddings = corpus_sums / np.maximum(corpus_counts[:, None], 1)
print(f'Corpus prototypes built: {corpus_prototype_embeddings.shape}')
print(f'Sample true label: {sample_true_label}')
# print(f'Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}')
print(f'Sample embedding shape: {sample_embedding.shape}')

Corpus prototypes built: (289, 128)
Sample true label: burg_spinnerlied.mid
Sample embedding shape: (128,)


In [19]:
# Load external test sequences
external_test_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_sequences.pkl')
print("External MIDI sequences loaded from pickle file.")
print(f"Shape of external test sequences DataFrame: {external_test_df.shape}")
print("Sample external test sequences:")
external_test_df.head()

External MIDI sequences loaded from pickle file.
Shape of external test sequences DataFrame: (203723, 2)
Sample external test sequences:


,sequence,label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid


In [20]:
X_test_external = np.stack(external_test_df['sequence'].to_numpy()).astype(np.float32)
X_test_external = (X_test_external - feature_mean) / feature_std

In [21]:
# Nearest-prototype retrieval on external test set
import tqdm

cnn_test_predictions = []
cnn_model.eval()

with torch.no_grad():
    for seq in tqdm.tqdm(X_test_external):
        seq_tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        emb        = cnn_model(seq_tensor).squeeze(0).cpu().numpy()
        sims       = np.array([cosine_similarity(emb, corpus_prototype_embeddings[c])
                               for c in range(num_classes)])
        pred_idx   = int(np.argmax(sims))
        cnn_test_predictions.append(label_encoder.inverse_transform([pred_idx])[0])

100%|██████████| 203723/203723 [12:36<00:00, 269.40it/s]


In [22]:
# Add predictions to external_test_df
external_test_df['cnn_predicted_label'] = cnn_test_predictions
print("Sample external test sequences with SiameseCNN predicted labels:")
external_test_df.head()

Sample external test sequences with SiameseCNN predicted labels:


,sequence,label,cnn_predicted_label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid,mz_330_1.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid,mz_330_1.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid,mz_330_1.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid,mz_330_1.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid,mz_330_1.mid


In [23]:
# Save external test predictions
results_path = '/content/drive/MyDrive/capstone_team_3/results/external_test_predictions_siamese_cnn_v1.pkl'
external_test_df[['label', 'cnn_predicted_label']].to_pickle(results_path)
print(f'External test predictions saved to {results_path}')

External test predictions saved to /content/drive/MyDrive/capstone_team_3/results/external_test_predictions_siamese_cnn_v1.pkl


In [24]:
# Load predictions back from pickle to verify
loaded_predictions_df = pd.read_pickle(
    '/content/drive/MyDrive/capstone_team_3/results/external_test_predictions_siamese_cnn_v1.pkl'
)
print("Loaded external test predictions from pickle file:")
print(loaded_predictions_df.head())

Loaded external test predictions from pickle file:
                       label cnn_predicted_label
0  Piano Sonata n12 K332.mid        mz_330_1.mid
1  Piano Sonata n12 K332.mid        mz_330_1.mid
2  Piano Sonata n12 K332.mid        mz_330_1.mid
3  Piano Sonata n12 K332.mid        mz_330_1.mid
4  Piano Sonata n12 K332.mid        mz_330_1.mid


In [27]:
!git clone https://github.com/mperumal-usd/capstone_team_3

Cloning into 'capstone_team_3'...
remote: Enumerating objects: 4922, done.
remote: Counting objects: 100% (192/192), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 4922 (delta 85), reused 85 (delta 25), pack-reused 4730 (from 3)
Receiving objects: 100% (4922/4922), 52.37 MiB | 13.38 MiB/s, done.
Resolving deltas: 100% (117/117), done.
Updating files: 100% (6337/6337), done.


In [34]:
!ls /content/capstone_team_3/MidiDatasets/TestingSamples/TestingReferences.csv

/content/capstone_team_3/MidiDatasets/TestingSamples/TestingReferences.csv


In [35]:
# Filter reference CSV to matched labels
reference_pd = pd.read_csv("/content/capstone_team_3/MidiDatasets/TestingSamples/TestingReferences.csv")
filtered_reference_pd = reference_pd[reference_pd['FileName'].isin(loaded_predictions_df['label'])]
print("Filtered reference MIDI sequences:")
print(filtered_reference_pd.shape[0])

Filtered reference MIDI sequences:
37


In [36]:
# Merge predictions with reference labels
merged_results_df = pd.merge(
    loaded_predictions_df, filtered_reference_pd,
    left_on='label', right_on='FileName', how='inner'
)
print("Merged predictions with reference labels:")
print(merged_results_df[['label', 'cnn_predicted_label', 'OriginalFileName']].head())

Merged predictions with reference labels:
                       label cnn_predicted_label OriginalFileName
0  Piano Sonata n12 K332.mid        mz_330_1.mid     mz_332_1.mid
1  Piano Sonata n12 K332.mid        mz_330_1.mid     mz_332_1.mid
2  Piano Sonata n12 K332.mid        mz_330_1.mid     mz_332_1.mid
3  Piano Sonata n12 K332.mid        mz_330_1.mid     mz_332_1.mid
4  Piano Sonata n12 K332.mid        mz_330_1.mid     mz_332_1.mid


In [37]:
# Save merged results
merged_results_df.to_pickle('/content/drive/MyDrive/capstone_team_3/results/siamese_cnn_merged_results.pkl')
print("Merged results saved to pickle file.")

Merged results saved to pickle file.


In [38]:
# External test accuracy
cnn_external_accuracy = accuracy_score(
    merged_results_df['OriginalFileName'],
    merged_results_df['cnn_predicted_label']
)
print(f'SiameseCNN External Test Accuracy: {cnn_external_accuracy:.4f}')

SiameseCNN External Test Accuracy: 0.0954


In [39]:
# Store model metadata — same structure as GRU notebook
final_epoch = cnn_history_df.iloc[-1]

model_meta_data = {
    "modelName" : "SiameseCNN",
    "checkpoint": cnn_model_save_path,
    "hyperParams": {
        "embedding_dim"  : cnn_embedding_dim,
        "conv_channels"  : [64, 128, 256],
        "kernel_sizes"   : [5, 3, 3],
        "dropout"        : cnn_dropout,
        "margin"         : cnn_margin,
        "mining_strategy": "semi-hard",
        "learning_rate"  : cnn_learning_rate,
        "epochs"         : cnn_epochs,
        "batch_size"     : cnn_batch_size,
        "optimizer"      : "Adam",
        "scheduler"      : "StepLR(step=2, gamma=0.5)",
        "loss"           : "TripletLoss",
    },
    "dataParams": {
        "input_dim"      : input_dim,
        "sequence_length": sequence_length,
        "num_classes"    : num_classes,
        "train_size"     : len(train_df),
        "test_size"      : len(test_df),
    },
    "results": {
        "final_train_loss"     : round(float(final_epoch['train_loss']),      4),
        "final_active_triplets": round(float(final_epoch['active_triplets']), 1),
        "final_test_accuracy"  : round(float(final_epoch['test_accuracy']),   4),
        "test_top1_accuracy"   : round(correct_at_k[1] / n_test,             4),
        "test_top3_accuracy"   : round(correct_at_k[3] / n_test,             4),
        "test_top5_accuracy"   : round(correct_at_k[5] / n_test,             4),
        "test_mrr"             : round(float(np.mean(recip_ranks)),           4),
        "external_test_accuracy": round(float(cnn_external_accuracy),         4),
    },
    "trainingHistory": cnn_history_df.to_dict(orient='records'),
}

print(json.dumps({k: v for k, v in model_meta_data.items() if k != 'trainingHistory'}, indent=2))
print(f"\nTraining history ({len(model_meta_data['trainingHistory'])} epochs) stored in model_meta_data['trainingHistory']")

{
  "modelName": "SiameseCNN",
  "checkpoint": "/content/drive/MyDrive/capstone_team_3/models/siamese_cnn_classifier.pth",
  "hyperParams": {
    "embedding_dim": 128,
    "conv_channels": [
      64,
      128,
      256
    ],
    "kernel_sizes": [
      5,
      3,
      3
    ],
    "dropout": 0.3,
    "margin": 0.2,
    "mining_strategy": "semi-hard",
    "learning_rate": 0.001,
    "epochs": 5,
    "batch_size": 64,
    "optimizer": "Adam",
    "scheduler": "StepLR(step=2, gamma=0.5)",
    "loss": "TripletLoss"
  },
  "dataParams": {
    "input_dim": 3,
    "sequence_length": 256,
    "num_classes": 289,
    "train_size": 504789,
    "test_size": 126198
  },
  "results": {
    "final_train_loss": 0.0087,
    "final_active_triplets": 1.9,
    "final_test_accuracy": 0.2516,
    "test_top1_accuracy": 0.2516,
    "test_top3_accuracy": 0.4808,
    "test_top5_accuracy": 0.6005,
    "test_mrr": 0.4118,
    "external_test_accuracy": 0.0954
  }
}

Training history (5 epochs) stored in mod

In [40]:
# Save metadata JSON to Google Drive
meta_save_path = '/content/drive/MyDrive/capstone_team_3/models/siamese_cnn_meta.json'
with open(meta_save_path, 'w') as f:
    json.dump(model_meta_data, f, indent=2)
print(f'Model metadata saved to {meta_save_path}')

Model metadata saved to /content/drive/MyDrive/capstone_team_3/models/siamese_cnn_meta.json
